# Build an MCP server, one decorator at a time

**Week 10 · Session 2 · Notebook 04**

We build the incident server piece by piece, testing each function as a plain Python
function first — then assemble the whole thing into `incident_server.py` at the end.

**Why plain functions first?** An MCP tool *is* a normal function with a decorator on
it. If the function is wrong, the decorator will not save you. Debug the logic where
debugging is easy, then add the protocol.

The finished file already exists in `mcp_server/`. This notebook shows how it got
there.

In [ ]:
%pip install -q mcp

In [ ]:
import re, sqlite3, sys
from datetime import datetime, timezone
from pathlib import Path

# Find the database whether you opened this from week10/ or week10/notebooks/
ROOT = next(p for p in [Path("mcp_server"), Path("../mcp_server")] if p.exists()).resolve()
DB = ROOT / "incidents.db"

if not DB.exists():
    print("Seeding the database first...")
    sys.path.insert(0, str(ROOT))
    import seed_data
    seed_data.build()

print("db:", DB)

---
## Step 0 — The query helper

Nothing MCP-specific. Every tool goes through this.

In [ ]:
def q(sql, args=()):
    """Query helper -- returns a list of dicts."""
    con = sqlite3.connect(DB)
    con.row_factory = sqlite3.Row
    try:
        return [dict(r) for r in con.execute(sql, args).fetchall()]
    finally:
        con.close()


print(q("SELECT name, team, tier FROM services"))
print()
print(q("SELECT COUNT(*) n FROM incidents"))

---
## Step 1 — The smallest possible tool

Start with the one that cannot fail: no arguments, no branching. It proves the wiring
before you add logic.

Three things become the MCP schema automatically:

| In Python | Becomes |
|---|---|
| function name | the tool name |
| **docstring** | the description the model reads — *this is prompt text* |
| type hints | the JSON Schema for parameters |

In [ ]:
from mcp.server.fastmcp import FastMCP

mcp = FastMCP(
    "incident-copilot",
    instructions=("Tools for investigating production incidents. Search past "
                  "incidents for prior art before proposing a fix, and read the "
                  "service runbook before recommending any action."),
)


@mcp.tool()
def list_services() -> str:
    """List every service this server knows about, with its owning team and tier.

    Call this first if you are unsure of an exact service name.
    """
    rows = q("SELECT name, team, tier, oncall FROM services ORDER BY tier, name")
    return "\n".join(f"{r['name']:<16} tier {r['tier']}  {r['team']:<10} {r['oncall']}"
                     for r in rows)


# The decorator registers it, but the function is still an ordinary function.
print(list_services())

> **Gotcha:** it is `@mcp.tool()` with parentheses, not `@mcp.tool`. Without them the
> decorator never runs and your server starts with zero tools — one of the three
> failures on the Inspector slide.

Let's see what the model will actually receive:

In [ ]:
import asyncio, json

async def show_schemas():
    for t in await mcp.list_tools():
        print(json.dumps({"name": t.name,
                          "description": t.description,
                          "parameters": t.inputSchema}, indent=2))

await show_schemas()

That `description` is your docstring, verbatim. **Read it back as if you were a
stranger.** If it does not say when to call the tool and what comes back, the model is
guessing.

---
## Step 2 — Arguments, optional and required

Type hints with defaults become optional parameters. Everything else is required.

In [ ]:
@mcp.tool()
def search_incidents(query: str, service: str = "", limit: int = 5) -> str:
    """Search past production incidents by keyword.

    Use this BEFORE proposing any fix -- someone has usually seen it before.
    Searches the title, summary, resolution and tags.

    Args:
        query: Keywords, e.g. "latency spike" or "connection pool". Not a question.
        service: Optional exact service name to narrow the search, e.g. "checkout-api".
        limit: Max results, 1-20. Default 5.

    Returns:
        Matching incidents with id, service, severity, status and a one-line summary.
        Call get_incident for the full record.
    """
    limit = max(1, min(int(limit), 20))          # never trust a generated number
    like = f"%{query.strip()}%"
    sql = ("SELECT id, service, severity, status, title, summary, tags, opened_at, "
           "duration_min FROM incidents WHERE (title LIKE ? OR summary LIKE ? OR "
           "resolution LIKE ? OR tags LIKE ?)")
    args = [like, like, like, like]
    if service.strip():
        sql += " AND service = ?"
        args.append(service.strip())
    sql += " ORDER BY opened_at DESC LIMIT ?"
    args.append(limit)

    rows = q(sql, args)
    if not rows:
        # A useful miss beats an empty string. Tell the model what to try next.
        return (f"No incidents matched '{query}'"
                + (f" for service '{service}'." if service else ".")
                + " Try broader keywords, or list_services to check the name.")

    out = [f"{len(rows)} match(es) for '{query}':"]
    for r in rows:
        out.append(
            f"\n{r['id']}  [{r['severity']}/{r['status']}]  {r['service']}  "
            f"({r['opened_at']}, {r['duration_min']}m)\n"
            f"  {r['title']}\n  {r['summary'][:180]}\n  tags: {r['tags']}"
        )
    return "\n".join(out)


print(search_incidents("connection pool", limit=2))
print()
print(search_incidents("zzzznothing"))        # the miss path matters

Two design choices worth naming:

1. **`limit` is clamped.** The model generated that number. It could be `10000`.
   Validate every argument before it reaches your database.
2. **The miss returns advice, not nothing.** `"Try broader keywords, or list_services"`
   gives the agent a recovery path. An empty string gives it nothing, and it will
   invent something instead.

---
## Step 3 — Shaping the output

A runbook is a long document. Returning the whole thing burns context for no reason —
so match on words, score the sections, and return only the best ones.

In [ ]:
@mcp.tool()
def search_runbooks(query: str, service: str = "") -> str:
    """Search the operational runbooks for a documented procedure.

    Use this to find the approved fix before improvising one. Matches on individual
    words, so "connection pool" also finds "connection-pool exhaustion".

    Args:
        query: What you need to do or diagnose, e.g. "connection pool".
        service: Optional exact service name to search only that runbook.

    Returns:
        The matching runbook sections, most relevant first, with their service.
    """
    words = [w for w in re.split(r"[^a-z0-9]+", query.lower()) if len(w) > 2]
    if not words:
        return "ERROR: give at least one search word of three or more characters."

    if service.strip():
        rows = q("SELECT service, body FROM runbooks WHERE service = ?",
                 (service.strip(),))
        if not rows:
            return (f"No runbook for '{service}'. Available: "
                    + ", ".join(r["name"] for r in q("SELECT name FROM runbooks")))
    else:
        rows = q("SELECT service, body FROM runbooks")

    def matches(word, text):
        # Crude stemming so "rotate" also finds "rotation".
        return word in text or (len(word) > 4 and word[:-2] in text)

    scored = []
    for r in rows:
        for section in r["body"].split("\n### "):
            low = section.lower()
            hits = sum(1 for w in words if matches(w, low))
            if hits:
                scored.append((hits, r["service"], section.strip()))
    if not scored:
        return (f"No runbook mentions any of {words}. Services with runbooks: "
                + ", ".join(r["name"] for r in q("SELECT name FROM runbooks")))

    scored.sort(key=lambda x: -x[0])
    out = []
    for hits, svc, section in scored[:3]:
        head = section.splitlines()[0]
        out.append(f"--- {svc} · matched {hits}/{len(words)} terms · {head}\n"
                   f"{section[:900]}")
    return "\n\n".join(out)


print(search_runbooks("connection pool")[:520])

**Shaping tool output in your own code is the cheapest context saving available.** No
model involved, no tokens spent, and you control exactly what gets through.

Compare: the full checkout-api runbook is ~1,400 characters. We returned one 900-char
section. Multiply that by a ten-step run.

---
## Step 4 — The write, with validation

`create_ticket` is the only tool that changes the world. Everything about it is
deliberately more careful.

In [ ]:
@mcp.tool()
def create_ticket(title: str, severity: str, service: str, body: str) -> str:
    """Create a follow-up ticket. THIS WRITES TO THE TRACKER.

    Only call this when the user has explicitly asked for a ticket to be created.
    Never create one on your own initiative to "record" or "track" something.

    Args:
        title: A short, specific title. Not a question.
        severity: One of SEV1, SEV2, SEV3.
        service: The affected service. Must be a known service name.
        body: What happened, what was tried, and what the follow-up action is.

    Returns:
        The new ticket id, or an error explaining what was wrong with the input.
    """
    # Validate EVERYTHING before touching the database. Each error names the
    # problem and confirms that nothing happened.
    sev = severity.strip().upper()
    if sev not in {"SEV1", "SEV2", "SEV3"}:
        return (f"ERROR: severity must be SEV1, SEV2 or SEV3 (got '{severity}'). "
                "Nothing created.")
    if not title.strip():
        return "ERROR: title is empty. Nothing created."
    known = {r["name"] for r in q("SELECT name FROM services")}
    if service.strip() not in known:
        return (f"ERROR: unknown service '{service}'. Known: {', '.join(sorted(known))}. "
                "Nothing created.")

    con = sqlite3.connect(DB)
    try:
        n = con.execute("SELECT COUNT(*) FROM tickets").fetchone()[0]
        tid = f"TKT-{2000 + n}"
        con.execute("INSERT INTO tickets VALUES (?,?,?,?,?,?,?)",
                    (tid, title.strip(), sev, service.strip(), body.strip(),
                     datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M"), "mcp-agent"))
        con.commit()
    finally:
        con.close()
    return f"Created {tid} [{sev}] on {service}: {title.strip()}"


print(create_ticket("x", "SEV9", "checkout-api", "y"))     # bad severity
print(create_ticket("x", "SEV2", "ghost", "y"))            # unknown service
print(create_ticket("", "SEV2", "checkout-api", "y"))      # empty title

Notice the docstring does guardrail work the code cannot: *"Only call this when the
user has explicitly asked"* and *"Never create one on your own initiative"*. The model
reads that on every turn.

That is not a substitute for the host's human gate — it is the first of several layers.
A determined model can still call it; the validation and the gate are what actually
stop it.

---
## Step 5 — Resources

Same idea, different control. A **resource** is app-controlled: the host decides when
to load it. The `{service}` in the URI becomes the function argument.

In [ ]:
@mcp.resource("runbook://{service}")
def runbook(service: str) -> str:
    """The full operational runbook for a service, as markdown."""
    rows = q("SELECT body FROM runbooks WHERE service = ?", (service,))
    if not rows:
        return f"# No runbook for '{service}'"
    return rows[0]["body"]


@mcp.resource("incidents://open")
def open_incidents() -> str:
    """Every incident that is not yet resolved."""
    rows = q("SELECT id, service, severity, title, opened_at FROM incidents "
             "WHERE status != 'resolved' ORDER BY severity, opened_at DESC")
    if not rows:
        return "No open incidents."
    return "\n".join(f"{r['id']}  [{r['severity']}]  {r['service']:<16} {r['title']}  "
                     f"(since {r['opened_at']})" for r in rows)


async def show_resources():
    print("templated:", [t.uriTemplate for t in await mcp.list_resource_templates()])
    print("static   :", [str(r.uri) for r in await mcp.list_resources()])
    print()
    print((await mcp.read_resource("incidents://open"))[0].content[:300])

await show_resources()

---
## Step 6 — Prompts

**User-controlled.** These appear as slash commands in the host app. A prompt returns
*text* — a reusable workflow you ship alongside your tools.

This is the primitive almost nobody builds, and it is where your domain expertise goes.

In [ ]:
@mcp.prompt()
def triage(symptom: str) -> str:
    """Triage a live symptom against history and runbooks."""
    return f"""We are seeing: {symptom}

Work in this order and show your reasoning:
1. list_services, then service_health on the most likely service.
2. search_incidents for this symptom -- has it happened before?
3. search_runbooks for the documented procedure.
4. Recommend the next concrete action, citing the incident id or runbook section
   it came from.

If the history and the runbook disagree, say so rather than picking one silently.
Do not create a ticket unless I ask."""


async def show_prompt():
    for p in await mcp.list_prompts():
        print(f"/{p.name}({', '.join(a.name for a in (p.arguments or []))})")
    got = await mcp.get_prompt("triage", {"symptom": "checkout 500s"})
    print()
    print(got.messages[0].content.text)

await show_prompt()

Read that prompt again — it encodes **how your team triages**. The tool list says what
is possible; the prompt says what good practice looks like. That is genuine
institutional knowledge, shipped as code.

---
## Step 7 — Assemble the file

Everything above is in one process, in a notebook. To make it a real server it needs
to be a file that runs `mcp.run()`.

The complete version is already at `mcp_server/incident_server.py` — it adds
`get_incident`, `service_health` and `incidents://recent`, all built the same way.

In [ ]:
print((ROOT / "incident_server.py").read_text()[:1500])
print("\n... full file is", len((ROOT / 'incident_server.py').read_text().splitlines()),
      "lines")

The entry point is the last three lines:

```python
if __name__ == "__main__":
    mcp.run(transport="stdio")
```

`stdio` means: the host launches this file as a subprocess and speaks JSON-RPC over
stdin/stdout.

> **The gotcha that will bite someone today:** with stdio, **stdout *is* the protocol
> channel**. A stray `print()` in your server corrupts the JSON-RPC stream and the
> connection dies with a confusing parse error. Log to `stderr`.

---
## Step 8 — Run it

Three ways, in increasing order of how much can go wrong:

In [ ]:
print(f"""
# 1. The Inspector -- a browser UI, no API key needed. Start here, always.
npx @modelcontextprotocol/inspector {sys.executable} {ROOT}/incident_server.py

# 2. A host app -- put this in claude_desktop_config.json, then FULLY restart it.
#    Absolute paths only; these apps do not expand ~.
{{
  "mcpServers": {{
    "incident-copilot": {{
      "command": "{sys.executable}",
      "args": ["{ROOT}/incident_server.py"]
    }}
  }}
}}

# 3. A real LLM -- see notebook 03, or:
python3 {ROOT}/client_demo.py
""")

---
## What you built

```
6 tools      list_services · search_incidents · get_incident
             search_runbooks · service_health · create_ticket  (the write)
3 resources  runbook://{service} · incidents://recent · incidents://open
2 prompts    /triage · /postmortem_draft
```

About 250 lines, no web framework, no boilerplate. And it now works in Claude Desktop,
in Cursor, and in any Python loop — without any of them knowing it exists.

---
## Exercises

**1. Add `incidents_by_tag(tag: str)`.** Then run notebook 03's step 1 again and watch
the client discover it with no client changes.

**2. Write a deliberately bad docstring** for one tool, restart, and run a question
through notebook 03 that needs it. Count the tool calls.

**3. Add a resource** `stats://summary` returning incident counts per service. Then
load it directly in notebook 03 instead of letting the model call a tool.

**4. Add a second prompt** `/weekly_report` that walks the model through summarising
the week's incidents.

**5. The real exercise.** Delete all of this and write a two-tool MCP server for a
system *you* actually use — your API, your database, your ticketing. Two tools is
enough. That is the week-11 homework, and it is where the value is.